In [1]:
import os
import json
import csv

def extract_condition(notebook_path):
    try:
        with open(notebook_path, 'r') as f:
            data = json.load(f)
            notes = data.get('notes', [])
            filtered_notes = [
                note.strip() for note in notes
                if note.strip().lower() not in ['l', 'nl']
            ]
            condition = " | ".join(filtered_notes) if filtered_notes else "Unlabeled"
            return condition
    except Exception as e:
        print(f"[!] Error reading notebook.json: {notebook_path}: {e}")
        return "Unknown"

def map_condition_to_number(condition):
    cond = condition.lower()
    if cond == "pc | light":
        return '1'
    elif cond == "pc | dark":
        return '2'
    else:
        return 0  # Or "" if you want to leave blank

def extract_laser_status(stimlog_path):
    try:
        with open(stimlog_path, 'r') as f:
            data = json.load(f)
            laser_flags = [trial.get("LaserOnOff", 0) for trial in data]
            return 1.0 if any(flag == 1 for flag in laser_flags) else 0.0
    except Exception as e:
        print(f"[!] Error reading stimlog.json: {stimlog_path}: {e}")
        return ""

def find_filtered_csv(session_dir):
    # Look for any CSV ending with _filtered.csv in the session directory (no deeper)
    try:
        for file in os.listdir(session_dir):
            if file.endswith('300000_el_filtered.csv'):
                return os.path.join(session_dir, file)
    except Exception as e:
        print(f"[!] Error searching for filtered CSV in {session_dir}: {e}")
    return ""

def map_trials_to_drop(condition):
    # Default rule: assume left_only unless you define another mapping
    return "left_only"

def scan_and_write_formatted_csv(rootdir, output_csv):
    rows = []

    for session_folder in os.listdir(rootdir):
        session_path = os.path.join(rootdir, session_folder)
        if not os.path.isdir(session_path):
            continue

        # Find filtered CSV in session folder (only here, not deeper)
        filtered_csv_path = find_filtered_csv(session_path)

        # Iterate over subsession folders inside this session
        subsession_folders = [d for d in os.listdir(session_path) if os.path.isdir(os.path.join(session_path, d))]
        for subsession in subsession_folders:
            subsession_path = os.path.join(session_path, subsession)

            notebook_path = os.path.join(subsession_path, 'notebook.json')
            stimlog_path = os.path.join(subsession_path, 'stimlog.json')

            if not (os.path.exists(notebook_path) and os.path.exists(stimlog_path)):
                continue

            condition = extract_condition(notebook_path)
            condition_number = map_condition_to_number(condition)
            laser = extract_laser_status(stimlog_path)
            trials_to_drop = map_trials_to_drop(condition)

            rows.append({
                'path': subsession_path,
                'Cond': condition_number,
                'FatalFlag': "",
                'Laser': laser,
                'posix_csv_path': filtered_csv_path,
                'trials_to_drop': trials_to_drop
            })

    if rows:
        with open(output_csv, 'w', newline='') as csvfile:
            fieldnames = ['path', 'Cond', 'FatalFlag', 'Laser', 'posix_csv_path', 'trials_to_drop']
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            writer.writeheader()
            for row in rows:
                writer.writerow(row)

        print(f"\n✅ CSV written to: {output_csv} ({len(rows)} entries)")
    else:
        print("\n⚠️ No valid data found. CSV not created.")



In [2]:

# Example usage
if __name__ == "__main__":
    root_directory = "/Volumes/Projects/PreyCapture/ZIActivation/ViralChR2_Activation/"  # CHANGE THIS
    output_csv = os.path.join(root_directory, "formatted_conditions_summary_viral.csv")
    scan_and_write_formatted_csv(root_directory, output_csv)


[!] Error reading stimlog.json: /Volumes/Projects/PreyCapture/ZIActivation/ViralChR2_Activation/2025-04-25_12-59-35_mouse-3356/2025-04-25_12-59-37_mouse-3356/stimlog.json: 'str' object has no attribute 'get'
[!] Error reading stimlog.json: /Volumes/Projects/PreyCapture/ZIActivation/ViralChR2_Activation/2025-04-26_13-07-16_mouse-3355/2025-04-26_13-07-20_mouse-3355/stimlog.json: 'str' object has no attribute 'get'
[!] Error reading stimlog.json: /Volumes/Projects/PreyCapture/ZIActivation/ViralChR2_Activation/2025-11-04_9-44-38_mouse-5093/2025-11-04_09-44-42_mouse-5093/stimlog.json: 'str' object has no attribute 'get'
[!] Error reading stimlog.json: /Volumes/Projects/PreyCapture/ZIActivation/ViralChR2_Activation/2025-10-21_18-06-56_mouse-5090/2025-10-21_18-06-59_mouse-5090/stimlog.json: 'str' object has no attribute 'get'
[!] Error reading stimlog.json: /Volumes/Projects/PreyCapture/ZIActivation/ViralChR2_Activation/2025-10-30_10-39-31_mouse-5091/2025-10-30_10-39-34_mouse-5091/stimlog.jso

In [3]:

# Example usage
if __name__ == "__main__":
    root_directory = "/Volumes/Projects/PreyCapture/ZIActivation/Zhao_VgatChR2_protocol/"  # CHANGE THIS
    output_csv = os.path.join(root_directory, "formatted_conditions_summary.csv")
    scan_and_write_formatted_csv(root_directory, output_csv)

KeyboardInterrupt: 

In [3]:
import csv
import re
import os
from datetime import datetime

def reorder_csv_by_folder_datetime(input_csv_path, output_csv_path, parent_dir):
    datetime_pattern = re.compile(r'(\d{4}-\d{2}-\d{2})[_T](\d{1,2}-\d{2}-\d{2})')

    rows_with_dt = []

    with open(input_csv_path, 'r', newline='') as f:
        reader = csv.DictReader(f)
        rows = list(reader)

    for row in rows:
        folder_path = row.get('path', '')  # assuming the folder path is in 'path' column
        if not folder_path:
            print("[!] Missing 'path' in row, skipping.")
            continue

        try:
            # Get relative path from parent_dir to folder_path
            rel_path = os.path.relpath(folder_path, parent_dir)
            parts = rel_path.split(os.sep)
            if not parts:
                print(f"[!] Empty relative folder path: {rel_path}")
                dt_obj = None
            else:
                first_folder = parts[0]

                match = datetime_pattern.search(first_folder)
                if match:
                    date_str = match.group(1)
                    time_str = match.group(2)

                    # Fix hour with leading zero if needed
                    hour_min_sec = time_str.split('-')
                    if len(hour_min_sec[0]) == 1:
                        hour_min_sec[0] = '0' + hour_min_sec[0]
                    fixed_time_str = ':'.join(hour_min_sec)

                    dt_str = f"{date_str} {fixed_time_str}"
                    try:
                        dt_obj = datetime.strptime(dt_str, "%Y-%m-%d %H:%M:%S")
                    except ValueError as e:
                        print(f"[!] Date parsing error for '{dt_str}' in folder '{first_folder}': {e}")
                        dt_obj = None
                else:
                    print(f"[!] No datetime found in folder name: {first_folder}")
                    dt_obj = None
        except Exception as e:
            print(f"[!] Error processing path '{folder_path}': {e}")
            dt_obj = None

        rows_with_dt.append((dt_obj, row))

    # Filter out rows without valid datetime if you want:
    rows_with_dt = [x for x in rows_with_dt if x[0] is not None]

    # Sort by datetime ascending (oldest first)
    rows_with_dt.sort(key=lambda x: x[0])

    # Write reordered CSV
    if rows_with_dt:
        with open(output_csv_path, 'w', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=rows_with_dt[0][1].keys())
            writer.writeheader()
            for _, row in rows_with_dt:
                writer.writerow(row)
        print(f"✅ Reordered CSV written to: {output_csv_path} ({len(rows_with_dt)} entries)")
    else:
        print("⚠️ No valid datetime entries found; no output written.")


In [6]:
parent_directory = "/Volumes/Projects/PreyCapture/ZI_A1Activation/"

reorder_csv_by_folder_datetime(
    input_csv_path="/Volumes/Projects/PreyCapture/ZI_A1Activation/formatted_conditions_summary.csv",
    output_csv_path="/Volumes/Projects/PreyCapture/ZI_A1Activation/formatted_conditions_summary_reordered.csv",
    parent_dir=parent_directory
)



✅ Reordered CSV written to: /Volumes/Projects/PreyCapture/ZI_A1Activation/formatted_conditions_summary_reordered.csv (4339 entries)


In [31]:
parent_directory = "/Volumes/Projects/PreyCapture/ZIActivation/Zhao_VgatChR2_protocol/"

reorder_csv_by_folder_datetime(
    input_csv_path="/Volumes/Projects/PreyCapture/ZIActivation/Zhao_VgatChR2_protocol/formatted_conditions_summary.csv",
    output_csv_path="/Volumes/Projects/PreyCapture/ZIActivation/Zhao_VgatChR2_protocol/formatted_conditions_summary_reordered_novice.csv",
    parent_dir=parent_directory
)


✅ Reordered CSV written to: /Volumes/Projects/PreyCapture/ZIActivation/Zhao_VgatChR2_protocol/formatted_conditions_summary_reordered_novice.csv (480 entries)


In [4]:
parent_directory = "/Volumes/Projects/PreyCapture/ZIActivation/ViralChR2_Activation/"

reorder_csv_by_folder_datetime(
    input_csv_path="/Volumes/Projects/PreyCapture/ZIActivation/ViralChR2_Activation/formatted_conditions_summary_viral.csv",
    output_csv_path="/Volumes/Projects/PreyCapture/ZIActivation/ViralChR2_Activation/formatted_conditions_summary_reordered_viral2.csv",
    parent_dir=parent_directory
)


✅ Reordered CSV written to: /Volumes/Projects/PreyCapture/ZIActivation/ViralChR2_Activation/formatted_conditions_summary_reordered_viral2.csv (2748 entries)
